In [1]:
import pandas as pd 

In [20]:
df = pd.read_csv('sentence_map.csv')

In [21]:
df

,Unnamed: 0,display_name,text_uuid,sentence_uuid,sentence_obj_in_text,translation,first_word_transcription,first_word_spelling,first_word_number,first_word_obj_in_text,line_number,side,column,transliteration_scraped
0,0,(Adana 237a) envelope,871e7671-e26b-4808-9acb-1c4a7e1ed2e6,f98879ad-d891-4602-8443-965baab6845d,3,Seal: trading station of Šalatiwar.,NaN,KIŠIB,1,5,1.00,1,1,NaN
1,1,(Adana 237s),09260971-37a9-4b91-aa30-ff59322598e5,e3444ecf-0276-4e5a-90cb-1e08e9d9f65e,3,From Imdī-ilum to Luzina and Puzur-Ištar:,NaN,um-ma,1,5,1.00,1,1,NaN
2,2,(HS 2931),a5abb045-e25a-469a-90b3-89a49091b7be,cb7f723d-6e52-454d-8ca2-50863cab81cb,10,"To the representatives Aššur-imittī, Imdī-ilum...",qibima,qí-bi₄-ma,8,11,3.00,1,1,NaN
3,3,(HS 2931),a5abb045-e25a-469a-90b3-89a49091b7be,5dc308e4-1033-4200-8b8b-6626d155539c,20,Ilī-uṣranni brings you 10 minas refined silver.,Ilī-uṣranni,ì-lí-uṣ-ra-ni,15,21,6.00,1,1,NaN
4,4,(HS 2931),a5abb045-e25a-469a-90b3-89a49091b7be,49ae0d28-4916-4744-b316-607432f9b091,23,Weigh it out to the one you recorded as my gua...,ana,a-na,17,25,8.00,1,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9777,9777,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,766d0d39-5767-4b98-bb71-d39fce93c908,20,(He will pay) 23? ḫamuštum weeks from the ḫamu...,ištu,iš-tù,11,22,4.00,1,1,ištu ḫamuštem ša Amur-Šamaš u Ikūnim ana 23 ḫa...
9778,9778,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,0220fef3-617c-4456-9dcf-d8a6e51bb44e,37,"[If he does not pay at full term, he will add ...",šumma,šu-ma,21,39,8.00,1,1,šumma ina ūmēšu mal'ūtem lā išqul (large break)
9779,9779,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,dbdfc4e9-420c-4a9f-891d-7147663384c4,53,...,ṭuppam,ṭup-pá-am,29,55,1.01,3,1,ṭuppam ša kunukkēšunu itadnam
9780,9780,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,c1cd791f-1451-4d2d-a2e8-70b559c4e895,67,If Uṣur-ša-Aššur does not hand over the tablet...,šumma,šu-ma,38,69,4.01,3,1,šumma ṭuppam ša kunukkīšu Uṣur-ša-Aššur lā itt...


In [48]:
import pandas as pd
import requests
response = requests.get('https://oare.byu.edu/api/v2/text_epigraphies/text/871e7671-e26b-4808-9acb-1c4a7e1ed2e6?forceAllowAdminView=false')

In [26]:
def get_transliteration_by_uuid(json_data, target_uuid):
    """
    Traverses the CDLI JSON structure to find a specific UUID 
    and returns its 'explicitSpelling'.
    """
    # Start the search with the root object
    # If json_data is a dict, put it in a list; otherwise assume it's a list
    stack = [json_data] if isinstance(json_data, dict) else json_data
    
    while stack:
        current = stack.pop()
        
        # If the current item is a dictionary, check it
        if isinstance(current, dict):
            # CHECK: Does this item match our target UUID?
            if current.get('uuid') == target_uuid:
                # Return the transliteration (explicitSpelling)
                return current.get('explicitSpelling')
            
            # CONTINUE SEARCH: Add children to the stack to be processed
            # The structure uses 'discourseUnits' at the top and 'units' below that
            if 'discourseUnits' in current:
                stack.extend(current['discourseUnits'])
            if 'units' in current:
                stack.extend(current['units'])
                
        # If the current item is a list (rare in this specific JSON but good for safety)
        elif isinstance(current, list):
            stack.extend(current)
            
    return None  # Return None if not found

In [28]:
import pandas as pd
import requests

# Placeholder for your JSON fetching logic
# You likely have a base URL structure like: https://cdli.mpiwg-berlin.mpg.de/artifacts/{text_uuid}/json
def fetch_json(text_uuid):
    url = f"https://oare.byu.edu/api/v2/text_epigraphies/text/{text_uuid}?forceAllowAdminView=false" 
    # Ideally, add headers or error handling here
    try:
        response = requests.get(url, headers={"Accept": "application/json"})
        if response.status_code == 200:
            return response.json()
    except:
        return None
    return None

# CACHING TIP: 
# Since multiple sentences share the same text_uuid, don't fetch the JSON 
# for every row. Fetch it once per text_uuid.

# 1. Get unique text_uuids and fetch their JSONs
unique_texts = df2['text_uuid'].unique()
# unique_texts=unique_texts[:10]
json_cache = {}

print(f"Fetching JSONs for {len(unique_texts)} unique texts...")
for t_uuid in unique_texts:
    # If you already have the JSON files locally, load them here instead of requesting
    json_data = fetch_json(t_uuid) 
    if json_data:
        json_cache[t_uuid] = json_data

# 2. Function to apply to the dataframe
def match_transliteration(row):
    t_uuid = row['text_uuid']
    s_uuid = row['sentence_uuid']
    
    if t_uuid in json_cache:
        return get_transliteration_by_uuid(json_cache[t_uuid], s_uuid)
    return None

# 3. Create the new column
df2['transliteration_scraped'] = df2.apply(match_transliteration, axis=1)

# Display result
print(df2[['sentence_uuid', 'transliteration_scraped']].head())

Fetching JSONs for 413 unique texts...
                          sentence_uuid  \
0  f98879ad-d891-4602-8443-965baab6845d   
1  e3444ecf-0276-4e5a-90cb-1e08e9d9f65e   
2  cb7f723d-6e52-454d-8ca2-50863cab81cb   
3  5dc308e4-1033-4200-8b8b-6626d155539c   
4  49ae0d28-4916-4744-b316-607432f9b091   

                             transliteration_scraped  
0                 KIŠIB wa-bar-tim ša ša-lá-tí-wa-ar  
1  um-ma Imdī-ilum-ma Anna Luzina ù Puzur-Ištar q...  
2                                             qibima  
3                           Ilī-uṣranni naš'akkunūti  
4             ana ša qātātīa ta-al-ta-áp-ta-ni šuqlā  


In [4]:
unique_texts = df['text_uuid'].unique()

In [12]:
df.to_csv('sentence_map.csv')

In [5]:
unique_texts

array(['871e7671-e26b-4808-9acb-1c4a7e1ed2e6',
       '09260971-37a9-4b91-aa30-ff59322598e5',
       'a5abb045-e25a-469a-90b3-89a49091b7be', ...,
       'f4a08131-02d9-47ed-3845-f63d73957762',
       '2d501011-3c78-9999-3fb4-2c5218cd1000',
       'ecb81e62-06b0-3ba0-6d4f-9372415c786d'], dtype=object)

In [10]:
df['transliteration_scraped'].isnull()

0        True
1        True
2        True
3        True
4        True
        ...  
9777    False
9778    False
9779    False
9780    False
9781    False
Name: transliteration_scraped, Length: 9782, dtype: bool

In [14]:
df.iloc[20]

Unnamed: 0                                                    20
display_name                                       (Kayseri 316)
text_uuid                   b6dfff9d-542c-44ba-83ae-d7b8d5d39e69
sentence_uuid               c5447c85-ac32-40a2-bddc-1878856395cc
sentence_obj_in_text                                          27
translation                        Year: the year after Šū-Niraḫ
first_word_transcription                                   līmum
first_word_spelling                                     li-mu-um
first_word_number                                             15
first_word_obj_in_text                                        29
line_number                                                  9.0
side                                                           1
column                                                         1
transliteration_scraped                                      NaN
Name: 20, dtype: object

In [47]:
# Get a list of the row indices where the value is NaN
nan_indices = df2[df2['transliteration_scraped'].isnull()].index.tolist()

len(nan_indices)
# Output will look like: [0, 1, 2, 3, 4, ...]

1912

In [31]:
df2.loc[nan_indices]

,Unnamed: 0,display_name,text_uuid,sentence_uuid,sentence_obj_in_text,translation,first_word_transcription,first_word_spelling,first_word_number,first_word_obj_in_text,line_number,side,column,transliteration_scraped
26,26,(Kayseri 90),9715151b-501a-4b99-874f-10eb770c6ae7,ad89ff7d-acd2-4fe2-9e8a-166d5f053436,3,"To Pūšu-kēn, Innāya, and Kiliya from Šū-Kūbum:",ana,a-na,1,5,1.0,1,1,None
27,27,(Kayseri 90),9715151b-501a-4b99-874f-10eb770c6ae7,df90307f-67af-4c83-88fc-003e55d8cf88,41,"My dear brothers, my share from my m. both to ...",u,ù,24,43,8.0,1,1,None
28,28,(Kayseri 90),9715151b-501a-4b99-874f-10eb770c6ae7,ac432f35-402b-4658-977a-f13164fa67d4,64,Take that which can be taken as cash as cash i...,kaspam,KÙ.BABBAR,38,66,14.0,3,1,None
29,29,(Kayseri 90),9715151b-501a-4b99-874f-10eb770c6ae7,20a64afc-4f93-4681-a9fb-13134d914808,81,"My dear brothers, my share from my m. should b...",ittaddima,i-ta-dí-ma,48,83,19.0,3,1,None
30,30,(Kayseri 90),9715151b-501a-4b99-874f-10eb770c6ae7,8b272b6c-7345-4e8a-a52a-36f7354304af,94,Take heed of my instructions.,ana,a-na,55,96,22.0,4,1,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8187,8187,Sadberk 13 (H.K. 1013-5542),31865db6-4ad9-4f0e-bb98-97e6db774995,b2d7b912-f3c4-4ad9-a8cf-6f2f2c8e4cbb,12,"2 talents 10 minas tin under seal, 10 minas ha...",bilat,GÚ,7,16,3.0,1,1,None
8188,8188,Sadberk 22 (H.K. 1012-5541),e977e211-2123-4081-85b6-6abbb815e810,175c4013-afa3-41b6-9b9b-2ec531624ae0,11,Abu-šalim owes Ilī-ālum 10 minas refined silver.,Ilī-ālum,ì-lí-a-lúm,7,12,4.0,1,1,None
8189,8189,Sadberk 22 (H.K. 1012-5541),e977e211-2123-4081-85b6-6abbb815e810,d1b57842-c436-45f1-9231-35a9fa6731f0,15,He will pay 15 ḫamuštum weeks from the ḫamuštu...,ištu,iš-tù,9,16,5.0,1,1,None
8190,8190,Sadberk 22 (H.K. 1012-5541),e977e211-2123-4081-85b6-6abbb815e810,6287dde6-7b86-432c-93dd-3469ef8b2c5f,38,"If he does not pay (on time), he will add 1 1/...",uṣṣab,ú-ṣa-áb,27,39,13.0,1,1,None


In [36]:
final_df = pd.concat([df, df2])

In [40]:
df

,Unnamed: 0,display_name,text_uuid,sentence_uuid,sentence_obj_in_text,translation,first_word_transcription,first_word_spelling,first_word_number,first_word_obj_in_text,line_number,side,column,transliteration_scraped
0,0,(Adana 237a) envelope,871e7671-e26b-4808-9acb-1c4a7e1ed2e6,f98879ad-d891-4602-8443-965baab6845d,3,Seal: trading station of Šalatiwar.,NaN,KIŠIB,1,5,1.00,1,1,NaN
1,1,(Adana 237s),09260971-37a9-4b91-aa30-ff59322598e5,e3444ecf-0276-4e5a-90cb-1e08e9d9f65e,3,From Imdī-ilum to Luzina and Puzur-Ištar:,NaN,um-ma,1,5,1.00,1,1,NaN
2,2,(HS 2931),a5abb045-e25a-469a-90b3-89a49091b7be,cb7f723d-6e52-454d-8ca2-50863cab81cb,10,"To the representatives Aššur-imittī, Imdī-ilum...",qibima,qí-bi₄-ma,8,11,3.00,1,1,NaN
3,3,(HS 2931),a5abb045-e25a-469a-90b3-89a49091b7be,5dc308e4-1033-4200-8b8b-6626d155539c,20,Ilī-uṣranni brings you 10 minas refined silver.,Ilī-uṣranni,ì-lí-uṣ-ra-ni,15,21,6.00,1,1,NaN
4,4,(HS 2931),a5abb045-e25a-469a-90b3-89a49091b7be,49ae0d28-4916-4744-b316-607432f9b091,23,Weigh it out to the one you recorded as my gua...,ana,a-na,17,25,8.00,1,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9777,9777,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,766d0d39-5767-4b98-bb71-d39fce93c908,20,(He will pay) 23? ḫamuštum weeks from the ḫamu...,ištu,iš-tù,11,22,4.00,1,1,ištu ḫamuštem ša Amur-Šamaš u Ikūnim ana 23 ḫa...
9778,9778,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,0220fef3-617c-4456-9dcf-d8a6e51bb44e,37,"[If he does not pay at full term, he will add ...",šumma,šu-ma,21,39,8.00,1,1,šumma ina ūmēšu mal'ūtem lā išqul (large break)
9779,9779,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,dbdfc4e9-420c-4a9f-891d-7147663384c4,53,...,ṭuppam,ṭup-pá-am,29,55,1.01,3,1,ṭuppam ša kunukkēšunu itadnam
9780,9780,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,c1cd791f-1451-4d2d-a2e8-70b559c4e895,67,If Uṣur-ša-Aššur does not hand over the tablet...,šumma,šu-ma,38,69,4.01,3,1,šumma ṭuppam ša kunukkīšu Uṣur-ša-Aššur lā itt...


In [45]:
final_df = final_df.sort_index()
final_df

,Unnamed: 0,display_name,text_uuid,sentence_uuid,sentence_obj_in_text,translation,first_word_transcription,first_word_spelling,first_word_number,first_word_obj_in_text,line_number,side,column,transliteration_scraped
45,45,(Nesr. C17),f6b5ebf1-50c3-4998-ad35-6ffda2f4dcf5,d9c82957-992c-4270-a00e-7cea9902db10,2,From Alāhum and Aššur-idī to Aššur-nādā and Šū...,umma,um-ma,1,4,1.00,1,1,umma Al-aḫum-ma Aššur-idī-ma ana Aššur-nādā u ...
46,46,(Nesr. C17),f6b5ebf1-50c3-4998-ad35-6ffda2f4dcf5,c0f7ea59-926c-4fd4-8703-9b5f4db8fcd6,15,(specifically) to Aššur-nādā:,ana,a-na,9,17,4.00,1,1,ana Aššur-nādā qibima
47,47,(Nesr. C17),f6b5ebf1-50c3-4998-ad35-6ffda2f4dcf5,e3c8821e-cae1-4709-8714-cae2c702a847,20,the same day that you hear the letter Šū-Aššur...,ina,i-na,12,21,5.00,1,1,ina ūmem ša ṭuppam tašammeāni 1 bilat AN.NA ku...
48,48,(Nesr. C17),f6b5ebf1-50c3-4998-ad35-6ffda2f4dcf5,b36ff976-b503-4f6c-82e6-ee8e2dead9b8,46,Why is it that you say:,miššu,mì-šu,32,47,11.00,1,1,miššu ša umma attāma
49,49,(Nesr. C17),f6b5ebf1-50c3-4998-ad35-6ffda2f4dcf5,b6fbdacb-fada-4e1b-aec0-d46706c6815f,53,"""You have seized my silver""?",taṣbat,ta-aṣ-ba-at,37,54,12.00,1,1,taṣbat
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9777,9777,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,766d0d39-5767-4b98-bb71-d39fce93c908,20,(He will pay) 23? ḫamuštum weeks from the ḫamu...,ištu,iš-tù,11,22,4.00,1,1,ištu ḫamuštem ša Amur-Šamaš u Ikūnim ana 23 ḫa...
9778,9778,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,0220fef3-617c-4456-9dcf-d8a6e51bb44e,37,"[If he does not pay at full term, he will add ...",šumma,šu-ma,21,39,8.00,1,1,šumma ina ūmēšu mal'ūtem lā išqul (large break)
9779,9779,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,dbdfc4e9-420c-4a9f-891d-7147663384c4,53,...,ṭuppam,ṭup-pá-am,29,55,1.01,3,1,ṭuppam ša kunukkēšunu itadnam
9780,9780,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,c1cd791f-1451-4d2d-a2e8-70b559c4e895,67,If Uṣur-ša-Aššur does not hand over the tablet...,šumma,šu-ma,38,69,4.01,3,1,šumma ṭuppam ša kunukkīšu Uṣur-ša-Aššur lā itt...


In [43]:
final_df_cleaned

,Unnamed: 0,display_name,text_uuid,sentence_uuid,sentence_obj_in_text,translation,first_word_transcription,first_word_spelling,first_word_number,first_word_obj_in_text,line_number,side,column,transliteration_scraped
45,45,(Nesr. C17),f6b5ebf1-50c3-4998-ad35-6ffda2f4dcf5,d9c82957-992c-4270-a00e-7cea9902db10,2,From Alāhum and Aššur-idī to Aššur-nādā and Šū...,umma,um-ma,1,4,1.00,1,1,umma Al-aḫum-ma Aššur-idī-ma ana Aššur-nādā u ...
46,46,(Nesr. C17),f6b5ebf1-50c3-4998-ad35-6ffda2f4dcf5,c0f7ea59-926c-4fd4-8703-9b5f4db8fcd6,15,(specifically) to Aššur-nādā:,ana,a-na,9,17,4.00,1,1,ana Aššur-nādā qibima
47,47,(Nesr. C17),f6b5ebf1-50c3-4998-ad35-6ffda2f4dcf5,e3c8821e-cae1-4709-8714-cae2c702a847,20,the same day that you hear the letter Šū-Aššur...,ina,i-na,12,21,5.00,1,1,ina ūmem ša ṭuppam tašammeāni 1 bilat AN.NA ku...
48,48,(Nesr. C17),f6b5ebf1-50c3-4998-ad35-6ffda2f4dcf5,b36ff976-b503-4f6c-82e6-ee8e2dead9b8,46,Why is it that you say:,miššu,mì-šu,32,47,11.00,1,1,miššu ša umma attāma
49,49,(Nesr. C17),f6b5ebf1-50c3-4998-ad35-6ffda2f4dcf5,b6fbdacb-fada-4e1b-aec0-d46706c6815f,53,"""You have seized my silver""?",taṣbat,ta-aṣ-ba-at,37,54,12.00,1,1,taṣbat
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9777,9777,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,766d0d39-5767-4b98-bb71-d39fce93c908,20,(He will pay) 23? ḫamuštum weeks from the ḫamu...,ištu,iš-tù,11,22,4.00,1,1,ištu ḫamuštem ša Amur-Šamaš u Ikūnim ana 23 ḫa...
9778,9778,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,0220fef3-617c-4456-9dcf-d8a6e51bb44e,37,"[If he does not pay at full term, he will add ...",šumma,šu-ma,21,39,8.00,1,1,šumma ina ūmēšu mal'ūtem lā išqul (large break)
9779,9779,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,dbdfc4e9-420c-4a9f-891d-7147663384c4,53,...,ṭuppam,ṭup-pá-am,29,55,1.01,3,1,ṭuppam ša kunukkēšunu itadnam
9780,9780,VS 26 99 (VAT 9268),ecb81e62-06b0-3ba0-6d4f-9372415c786d,c1cd791f-1451-4d2d-a2e8-70b559c4e895,67,If Uṣur-ša-Aššur does not hand over the tablet...,šumma,šu-ma,38,69,4.01,3,1,šumma ṭuppam ša kunukkīšu Uṣur-ša-Aššur lā itt...


In [46]:
df2

,Unnamed: 0,display_name,text_uuid,sentence_uuid,sentence_obj_in_text,translation,first_word_transcription,first_word_spelling,first_word_number,first_word_obj_in_text,line_number,side,column,transliteration_scraped
0,0,(Adana 237a) envelope,871e7671-e26b-4808-9acb-1c4a7e1ed2e6,f98879ad-d891-4602-8443-965baab6845d,3,Seal: trading station of Šalatiwar.,NaN,KIŠIB,1,5,1.0,1,1,KIŠIB wa-bar-tim ša ša-lá-tí-wa-ar
1,1,(Adana 237s),09260971-37a9-4b91-aa30-ff59322598e5,e3444ecf-0276-4e5a-90cb-1e08e9d9f65e,3,From Imdī-ilum to Luzina and Puzur-Ištar:,NaN,um-ma,1,5,1.0,1,1,um-ma Imdī-ilum-ma Anna Luzina ù Puzur-Ištar q...
2,2,(HS 2931),a5abb045-e25a-469a-90b3-89a49091b7be,cb7f723d-6e52-454d-8ca2-50863cab81cb,10,"To the representatives Aššur-imittī, Imdī-ilum...",qibima,qí-bi₄-ma,8,11,3.0,1,1,qibima
3,3,(HS 2931),a5abb045-e25a-469a-90b3-89a49091b7be,5dc308e4-1033-4200-8b8b-6626d155539c,20,Ilī-uṣranni brings you 10 minas refined silver.,Ilī-uṣranni,ì-lí-uṣ-ra-ni,15,21,6.0,1,1,Ilī-uṣranni naš'akkunūti
4,4,(HS 2931),a5abb045-e25a-469a-90b3-89a49091b7be,49ae0d28-4916-4744-b316-607432f9b091,23,Weigh it out to the one you recorded as my gua...,ana,a-na,17,25,8.0,1,1,ana ša qātātīa ta-al-ta-áp-ta-ni šuqlā
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8187,8187,Sadberk 13 (H.K. 1013-5542),31865db6-4ad9-4f0e-bb98-97e6db774995,b2d7b912-f3c4-4ad9-a8cf-6f2f2c8e4cbb,12,"2 talents 10 minas tin under seal, 10 minas ha...",bilat,GÚ,7,16,3.0,1,1,None
8188,8188,Sadberk 22 (H.K. 1012-5541),e977e211-2123-4081-85b6-6abbb815e810,175c4013-afa3-41b6-9b9b-2ec531624ae0,11,Abu-šalim owes Ilī-ālum 10 minas refined silver.,Ilī-ālum,ì-lí-a-lúm,7,12,4.0,1,1,None
8189,8189,Sadberk 22 (H.K. 1012-5541),e977e211-2123-4081-85b6-6abbb815e810,d1b57842-c436-45f1-9231-35a9fa6731f0,15,He will pay 15 ḫamuštum weeks from the ḫamuštu...,ištu,iš-tù,9,16,5.0,1,1,None
8190,8190,Sadberk 22 (H.K. 1012-5541),e977e211-2123-4081-85b6-6abbb815e810,6287dde6-7b86-432c-93dd-3469ef8b2c5f,38,"If he does not pay (on time), he will add 1 1/...",uṣṣab,ú-ṣa-áb,27,39,13.0,1,1,None


In [49]:
# Create a new dataframe without the NaN rows
df3 = df2.drop(nan_indices)


In [51]:
df3.to_csv('25_sentence_map.csv')

In [56]:
import pandas as pd

train = pd.read_csv("train.csv")
sent = pd.read_csv("Sentences_Oare_FirstWord_LinNum.csv")

missing_oare_ids = train.loc[
    ~train["oare_id"].isin(sent["text_uuid"]),
    "oare_id"
]

print(missing_oare_ids)


1       0064939c-59b9-4448-a63d-34612af0a1b5
2       0073f2c0-524c-4bbf-915a-8c1772a4fb98
4       00aa1c55-c80c-4346-a159-73ad43ab0ff7
5       00f0d841-eb7a-46f8-86fc-bf9fd7d52cbf
6       0123a9b9-e81e-4d7a-a79b-10e7c0aacbb9
                        ...                 
1555    feb2c219-9f9f-4569-9cd3-845bf8cf2a4c
1556    ff3208e4-8ab8-4368-b4df-7b80afa5bc32
1558    ff5747a4-af8a-4100-a906-a2660ae72606
1559    ff777871-97ce-4bfc-bdfb-73352868944d
1560    ff9442fd-9e7d-449c-a2d6-0cc35921cd65
Name: oare_id, Length: 1308, dtype: object


In [57]:
missing_rows = train[~train["oare_id"].isin(sent["text_uuid"])]

print(missing_rows)


                                   oare_id  \
1     0064939c-59b9-4448-a63d-34612af0a1b5   
2     0073f2c0-524c-4bbf-915a-8c1772a4fb98   
4     00aa1c55-c80c-4346-a159-73ad43ab0ff7   
5     00f0d841-eb7a-46f8-86fc-bf9fd7d52cbf   
6     0123a9b9-e81e-4d7a-a79b-10e7c0aacbb9   
...                                    ...   
1555  feb2c219-9f9f-4569-9cd3-845bf8cf2a4c   
1556  ff3208e4-8ab8-4368-b4df-7b80afa5bc32   
1558  ff5747a4-af8a-4100-a906-a2660ae72606   
1559  ff777871-97ce-4bfc-bdfb-73352868944d   
1560  ff9442fd-9e7d-449c-a2d6-0cc35921cd65   

                                        transliteration  \
1                  1 TÚG ša qá-tim i-tur₄-DINGIR il₅-qé   
2     TÚG u-la i-dí-na-ku-um i-tù-ra-ma 9 GÍN KÙ.BABBAR   
4     um-ma šu-ku-tum-ma a-na IŠTAR-lá-ma-sí ù ni-ta...   
5     um-ma šu-ta-mu-zi e-lá-a en-um-a-šùr ù lá-ma-s...   
6     KIŠIB a-lá-ḫi-im KIŠIB a-li-li KIŠIB a-bi₄-lá ...   
...                                                 ...   
1555  a-na (d)IM-ba-ni … Ì-lí-iš-t

In [59]:
resources = pd.read_csv('resources.csv')

In [62]:
resources

,Authors [separate with ; last name first],Year,Title,Additional info (i.e. journal),Topics,Language/dialect,Methods,Data,Project,URL,English abstract for non-English or N/A papers,University,Peer-reviewed,Type,OtherLinks,Review
0,"Gardin, Jean-Claude; Garelli, Paul",1961,Étude des établissements assyriens en Cappadoc...,"Annales: Economies, Societes, Civilisations; A...",Computational Analysis; SNA,Old Assyrian,Decision trees; Graph Modeling,NaN,Old Assyrian Text Project,NaN,(see Anderson 2018 for more info),NaN,yes,Research Paper,NaN,NaN
1,"Gardin, Jean-Claude",1965,Reconstructing an Economic Network in the Anci...,Studies in General Anthropology II,Computational Analysis; SNA,Old Assyrian,Decision trees; Graph Modeling,NaN,Old Assyrian Text Project,NaN,NaN,NaN,yes,Research Paper,NaN,NaN
2,"Parpola, Simo",1970,Neo-Assyrian Toponyms,Butzon & Bercker,Computer Generated Data,Neo-Assyrian,NaN,NaN,NaN,NaN,NaN,NaN,yes,Research Paper,NaN,NaN
3,"Buccellati, Giorgio",1977,The Old Babylonian Linguistic Analysis Project...,Proc. of the international conference on compu...,Project description,Old Babylonian,NaN,NaN,The Old Babylonian Linguistic Analysis Project,http://urkesh.org/EL2/Buccellati_1977_Old%20Ba...,NaN,NaN,yes,Research Paper,NaN,NaN
4,"Buccellati, Giorgio",1979,Comparative Graphemic Analysis of Old Babyloni...,"Schaeffer Festschrift Ugarit-Forschungen 11, N...",Computer Generated Data; Graphemic analysis,Old Babylonian,NaN,NaN,The Old Babylonian Linguistic Analysis Project,https://figshare.com/articles/journal%20contri...,NaN,NaN,yes,Research Paper,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
287,"Pottorf, Andrew",2025,Housing Data in Ur III Texts,Zenodo FactGrid Community,Dataset,Sumerian,Counts,https://zenodo.org/records/17563926,NaN,https://doi.org/10.5281/zenodo.17563926,NaN,NaN,no,Dataset,NaN,NaN
288,"Pottorf, Andrew",2025,Subsistence- and Tenant-Land Data in Ur III Um...,NaN,Dataset,Sumerian,Counts,https://zenodo.org/records/17517681,NaN,https://doi.org/10.5281/zenodo.17517681,NaN,NaN,no,Dataset,NaN,NaN
289,"Rattenborg, Rune; Pagé-Perron, Émilie",2025,The Cuneiform Digital Library Initiative: A Pr...,(Mar Shiprim and Zenodo),NaN,NaN,NaN,https://zenodo.org/records/15309738,NaN,https://doi.org/10.5281/zenodo.15309737,NaN,NaN,yes,Descriptive notes,NaN,NaN
290,"Riemenschneider, Frederick",2025,Beyond Base Predictors: Using LLMs to Resolve ...,Proceedings of the Second Workshop on Ancient ...,NaN,NaN,NaN,NaN,NaN,https://aclanthology.org/2025.alp-1.30/,NaN,NaN,yes,Research Paper,NaN,NaN


In [63]:
resources[resources['Title']== 'Glossed Hittite Texts with German Translation for Machine Learning']

,Authors [separate with ; last name first],Year,Title,Additional info (i.e. journal),Topics,Language/dialect,Methods,Data,Project,URL,English abstract for non-English or N/A papers,University,Peer-reviewed,Type,OtherLinks,Review
278,"Yavasan, Emma; Gordin, Shai",2024,Glossed Hittite Texts with German Translation ...,"Zenodo, Digital Ancient Near Eastern Studies N...",ML,"Hittite, German",Alignment,https://zenodo.org/records/14266302,NaN,https://doi.org/10.5281/zenodo.14266301,NaN,NaN,NaN,Dataset,NaN,NaN
